# Simple recursive network for sequence learning

In [1]:
import sys
from pathlib import Path
import numpy as np
from numpy.typing import ArrayLike
import torch
from typing import Callable, Sequence
from tqdm import trange


# import local packages
sys.path.append(str(Path().resolve().parents[0]))

from src.utils.logger import setup_logger, add_log_level
from src.utils.args import get_args
from src.data.formater import PandasLoader


In [2]:
args = get_args()
logger_level = "TRACE"


In [3]:
add_log_level("TRACE", 5)
logger = setup_logger(name="logger", level=logger_level)
logger.info("logger set to info level")
logger.trace("TRACE logger activated")



INFO - logger set to info level
TRACE - TRACE logger activated


In [4]:
loader = PandasLoader(args["input"], args["format"])
responses = loader.get()
logger.info(f"subjects response per trial: \n {responses}")
# converting to array of shape (subjects, responses)
responses = responses.to_numpy().T
logger.info(f"responses after transposition= {responses}")
# responses = np.expand_dims(responses, axis=-1)
# logger.trace(f"response after expansions on last dim={responses}")
# get set of responses values and their order of appearance(giving them a numerical id)
unique_vals, encoded = np.unique(responses.reshape(-1), return_inverse=True)
logger.info(f"unique values (set)={unique_vals}")
logger.info(f"encoded values={encoded}")
encoded = encoded.reshape(responses.shape)
logger.debug(f"final encoded shape (subjests, responses)={encoded.shape}")



INFO - subjects response per trial: 
 Subject    1  2  3  4  5  6  7  8  9  10  ... 51 52 53 54 55 56 57 58 59 60
TrialIndex                                ...                              
0           a  a  a  b  c  a  b  c  a  a  ...  a  a  a  b  c  a  b  c  b  b
1           J  Q  P  M  T  K  I  S  J  Q  ...  J  Q  P  M  T  K  I  S  L  H
2           d  d  d  e  f  d  e  f  d  d  ...  d  d  d  e  f  d  e  f  e  e
3           a  b  b  c  c  b  c  b  a  b  ...  a  b  b  c  c  b  c  b  b  c
4           S  I  O  Y  I  £  Q  H  S  I  ...  S  I  O  Y  I  £  Q  H  K  K
...        .. .. .. .. .. .. .. .. .. ..  ... .. .. .. .. .. .. .. .. .. ..
643         P  N  H  Y  £  G  H  M  P  N  ...  P  N  H  Y  £  G  H  M  H  W
644         d  e  d  e  d  d  f  f  d  e  ...  d  e  d  e  d  d  f  f  f  f
645         c  c  c  c  a  a  c  b  c  c  ...  c  c  c  c  a  a  c  b  b  c
646         O  I  K  ù  I  ù  &  Z  O  I  ...  O  I  K  ù  I  ù  &  Z  N  H
647         f  f  f  f  d  d  f  e  f  f  ...  f  

In [5]:
def make_uniform_tensor(
    extremum: tuple[float, float], shape: Sequence[int], grad: bool
):
    t = torch.empty(*shape)
    t.uniform_(*extremum)
    if grad:
        t.requires_grad_()
    return t


![test](https://web.stanford.edu/group/pdplab/pdphandbook/srn_net.png)

A SRN is a simplified RNN. The output of the hidden layer is fed back as input to the hidden layer at the
next time step. The output of the hidden layer is also used to compute the output of the network.

![SRN basic architecture](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.researchgate.net%2Fpublication%2F361380872%2Ffigure%2Ffig1%2FAS%3A1169080920875058%401655742020827%2FSchematic-diagram-of-Elman-network-structure-in-simple-recurrent-neural-network.jpg&f=1&nofb=1&ipt=1d5a36f3ef48ba69e88b9f96a9176f2e1ed8232b2019b9d3f7f7335e1ee85f1d)
```mermaid
graph TB;
hidden --> context
input --> hidden
context --> hidden
hidden --> output
```

In [6]:
class SRN_subject:
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        activation: list[Callable] = [torch.nn.Tanh],
        lr: float = 0.05,
        initial_w_unif: tuple[float, float] = (-0.1, 0.1),
        loss_fn: Callable = torch.nn.MSELoss()
    ):

        self.lr = lr
        self.activation = activation
        self.loss_fn = loss_fn

        self.Wxh = make_uniform_tensor(
            extremum=initial_w_unif, shape=[hidden_size, input_size + hidden_size], grad=True
        )
        
        self.Why = make_uniform_tensor(
            extremum=initial_w_unif, shape=[output_size, hidden_size], grad=True
        )

        self.context = torch.zeros(hidden_size)

    def forward(self, x: torch.Tensor, y: torch.Tensor | None = None):
        # input of hidden layer: input concatenated with previous output of the hidden layer
        x_cat_context = torch.cat([x, self.context], dim=0)

        h = activation[0](self.Wxh @ x_cat_context)
        self.context = h.detach()  # detach the context from the computational graph to prevent backprop through time
        y = activation[1](self.Why @ h)
        return y

    def backprop(self, y_pred, y):


        loss = self.loss_fn(y_pred, y)
        loss.backward()

        if self.Wxh.grad is None or self.Why.grad is None:
            e = RuntimeError("gradient missing")
            logger.error(e)
            raise e

        with torch.no_grad():
            self.Wxh -= self.lr * self.Wxh.grad
            self.Why -= self.lr * self.Why.grad
            self.Wxh.grad.zero_()
            self.Why.grad.zero_()

        return loss




Question:
For the backpropagation, when we learn patern on on a screen, how do we see the cases with no stimulus? making them -1 (opposite to the cell with stimulus 1).

In [ ]:

input_size = len(unique_vals)
output_size = len(unique_vals)


### hyperparameters